# Infinite-Width Networks

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 11.03 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/04_infinite_width_networks.ipynb)

---

## 🎯 Learning Objective

Study mean-field theory, signal propagation, and the edge-of-chaos regime that governs infinite-width network behavior.

---

## 📐 Theory

`11.03`'s NTK theory describes what happens to the network's *gradients* in the infinite-width
limit. This closing notebook studies the simpler, earlier question that theory was originally
built to answer: what happens to the *signal itself* (activations) as it propagates forward
through an increasingly deep, increasingly wide network at initialization — and why does the
answer determine whether that network can be trained at all.

### Mean-field theory: tracking one number instead of a whole activation vector

In an infinitely wide layer, the activations become (by an argument reminiscent of the Central
Limit Theorem, recall `03.06`) well-approximated by i.i.d. Gaussian random variables. This
**mean-field** simplification means you don't need to track a whole width-$n$ activation vector
through the network — just a couple of scalar statistics (its variance, and its correlation with
another input's activation) evolving layer-to-layer according to a deterministic recursion. This
notebook verifies that recursion's key qualitative predictions empirically, on genuinely wide
(but finite) simulated networks, rather than deriving the recursion's exact closed form.

### Signal propagation: variance across depth

For a layer $z = Wx + b$ with $W_{ij}\sim\mathcal{N}(0,\sigma_w^2/n)$ i.i.d. (the $1/n$ scaling
is exactly `11.03`'s NTK-parameterization idea, applied to keep pre-activation variance from
blowing up with width), the pre-activation variance evolves as $q^{(\ell+1)} = \sigma_w^2\,
\mathbb{E}[\phi(z)^2] + \sigma_b^2$ for activation $\phi$. Whether this recursion has a stable
fixed point — and whether that fixed point is reached quickly or slowly — determines whether
signal *vanishes* (variance $\to 0$, every unit's activation collapses to the same constant,
destroying all information) or *explodes* (variance $\to\infty$, numerical overflow) as depth
grows. Below, we verify He initialization's specific value $\sigma_w^2=2$ (for ReLU) is
precisely the value that keeps variance roughly constant across depth, while noticeably smaller
or larger values cause exactly the vanishing/exploding behavior just described.

### The correlation map and the order/chaos phase transition

A second, independent recursion tracks $c^{(\ell)}$, the correlation between two *different*
inputs' activations at layer $\ell$. For a saturating activation like $\tanh$, this recursion
has two qualitatively different long-run behaviors depending on $\sigma_w^2$: an **ordered
phase** where $c^{(\ell)}\to 1$ as depth grows (any two distinct inputs become
indistinguishable — the network collapses all information), and a **chaotic phase** where
$c^{(\ell)}$ converges to some value below 1 (inputs stay distinguishable, but nearby inputs'
outputs decorrelate unpredictably with depth). Between them sits the **edge of chaos**: the
critical $\sigma_w^2$ where the correlation map's fixed point at $c=1$ transitions from
attracting to repelling. Below, we map out this transition directly by propagating pairs of
correlated random vectors through networks at several $\sigma_w^2$ values and plotting how their
correlation evolves with depth — the qualitative signature of order vs. chaos is visible without
needing the recursion's exact closed form.

### Dynamical isometry (brief note)

**Dynamical isometry** is a stronger condition than merely "the edge of chaos": it asks that the
*entire spectrum* of the input-output Jacobian be close to 1 (not just its mean squared value),
which requires careful joint control of both the weight distribution and the activation function
together (e.g. orthogonal weight init) — full treatment is beyond this notebook's scope, but the
underlying motivation is identical: well-conditioned signal propagation, at every layer, not
just on average.

### Why this determines trainability

A network stuck in the ordered phase has activations (and, by a closely related argument,
gradients) that shrink geometrically with depth — the same vanishing-gradient phenomenon `06`
introduced concretely for deep networks. Below, we close the loop: train real (finite but deep)
networks at several $\sigma_w^2$ values and confirm directly that the one deep in the ordered
phase fails to learn at all (loss stuck near its initial value, near-zero gradient), while
initializations at or past the edge of chaos train successfully — turning "edge of chaos" from
an abstract phase-transition concept into a directly observable trainability threshold.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Propagating two initially weakly-correlated random vectors through a `tanh` network at several
values of $\sigma_w^2$ and tracking how their correlation evolves with depth makes the
order/chaos phase transition directly visible: small $\sigma_w^2$ drives correlation to 1
(information collapse), large $\sigma_w^2$ keeps it well below 1.

In [ ]:
rng_viz = np.random.default_rng(0)

def propagate_correlation(sigma_w_sq, sigma_b_sq, n_layers, width, activation, c0, n_trials, rng):
    correlations = []
    for trial in range(n_trials):
        x1 = rng.normal(size=width)
        x1 = x1 / np.linalg.norm(x1) * np.sqrt(width)
        noise = rng.normal(size=width)
        noise = noise / np.linalg.norm(noise) * np.sqrt(width)
        x2 = c0 * x1 + np.sqrt(max(1 - c0**2, 0)) * noise
        layer_corrs = [np.corrcoef(x1, x2)[0, 1]]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z1, z2 = W @ x1 + b, W @ x2 + b
            x1, x2 = activation(z1), activation(z2)
            layer_corrs.append(np.corrcoef(z1, z2)[0, 1])
        correlations.append(layer_corrs)
    return np.mean(correlations, axis=0)

width, n_layers, n_trials = 250, 20, 25
sigma_w_values = [0.8, 1.05, 1.3, 3.0]
colors = ["#C44E52", "#DD8452", "#55A868", "#4C72B0"]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
final_corrs = {}
for sigma_w_sq, color in zip(sigma_w_values, colors):
    corr_traj = propagate_correlation(sigma_w_sq, 0.05, n_layers, width, np.tanh, c0=0.3, n_trials=n_trials, rng=rng_viz)
    final_corrs[sigma_w_sq] = corr_traj[-1]
    ax.plot(corr_traj, color=color, lw=2, marker='o', ms=4, label=f"$\\sigma_w^2$={sigma_w_sq}")

ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel("layer depth"); ax.set_ylabel("correlation between two inputs' activations")
ax.set_title("Order/chaos phase transition in a deep tanh network:\nsmall $\\sigma_w^2$ collapses correlation to 1, large keeps it well below")
ax.legend()
plt.show()

print("Final correlation (after", n_layers, "layers), starting from c0=0.3:")
for sigma_w_sq in sigma_w_values:
    print(f"  sigma_w^2={sigma_w_sq}: final correlation = {final_corrs[sigma_w_sq]:.4f}")
print(f"\nSmall sigma_w^2 (ordered phase) drives correlation toward 1 -- any two DIFFERENT")
print(f"inputs become indistinguishable by the last layer. Large sigma_w^2 (chaotic phase)")
print(f"keeps correlation well below 1, preserving SOME ability to distinguish inputs.")

## 🐍 Implementation from Scratch

We verify He initialization's specific value keeps ReLU pre-activation variance roughly
constant across depth (while smaller/larger values cause genuine vanishing/exploding), then
quantify the order/chaos correlation map's phase transition precisely by checking how many
layers it takes to reach near-total correlation collapse at each $\sigma_w^2$.

In [ ]:
rng = np.random.default_rng(0)

def propagate_variance(sigma_w_sq, sigma_b_sq, n_layers, width, activation, x0_var, n_trials, rng):
    # Tracks a SINGLE unit's pre-activation variance across layers, for a wide network at
    # initialization -- the empirical quantity mean-field theory predicts a recursion for.
    variances = []
    for trial in range(n_trials):
        x = rng.normal(scale=np.sqrt(x0_var), size=width)
        layer_vars = [np.var(x)]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z = W @ x + b
            x = activation(z)
            layer_vars.append(np.var(z))
        variances.append(layer_vars)
    return np.mean(variances, axis=0)

relu = lambda z: np.maximum(z, 0)
width, n_layers, n_trials = 300, 25, 30

var_he = propagate_variance(2.0, 0.0, n_layers, width, relu, 1.0, n_trials, rng)   # He's own sigma_w^2=2 for ReLU
var_low = propagate_variance(0.5, 0.0, n_layers, width, relu, 1.0, n_trials, rng)
var_high = propagate_variance(8.0, 0.0, n_layers, width, relu, 1.0, n_trials, rng)

print("Pre-activation variance across depth (every 5 layers):")
print(f"  He init (sigma_w^2=2.0, the textbook ReLU value): {np.round(var_he[::5], 4)}")
print(f"  Too-small (sigma_w^2=0.5):                        {np.round(var_low[::5], 8)}")
print(f"  Too-large (sigma_w^2=8.0):                        {np.round(var_high[::5], 2)}")
print(f"\nHe's value keeps variance in a stable, bounded range across 25 layers; the too-small")
print(f"value drives it to (numerically) zero, and the too-large value drives it to explode")
print(f"by orders of magnitude -- exactly the vanishing/exploding-signal failure modes this")
print(f"specific choice of sigma_w^2 is designed to avoid.")

# --- Quantify the order/chaos correlation map's phase transition ---
def propagate_correlation(sigma_w_sq, sigma_b_sq, n_layers, width, activation, c0, n_trials, rng):
    correlations = []
    for trial in range(n_trials):
        x1 = rng.normal(size=width)
        x1 = x1 / np.linalg.norm(x1) * np.sqrt(width)
        noise = rng.normal(size=width)
        noise = noise / np.linalg.norm(noise) * np.sqrt(width)
        x2 = c0 * x1 + np.sqrt(max(1 - c0**2, 0)) * noise
        layer_corrs = [np.corrcoef(x1, x2)[0, 1]]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z1, z2 = W @ x1 + b, W @ x2 + b
            x1, x2 = activation(z1), activation(z2)
            layer_corrs.append(np.corrcoef(z1, z2)[0, 1])
        correlations.append(layer_corrs)
    return np.mean(correlations, axis=0)

print("\nCorrelation-map phase transition, tanh network, starting from c0=0.3:")
for sigma_w_sq in [0.8, 1.05, 1.3, 3.0]:
    corr_traj = propagate_correlation(sigma_w_sq, 0.05, n_layers=20, width=250, activation=np.tanh,
                                       c0=0.3, n_trials=25, rng=rng)
    depth_to_99 = next((i for i, c in enumerate(corr_traj) if c > 0.99), None)
    print(f"  sigma_w^2={sigma_w_sq}: final correlation (depth 20) = {corr_traj[-1]:.4f}, "
          f"reaches >0.99 by layer {depth_to_99 if depth_to_99 is not None else 'never'}")
print(f"\nSmaller sigma_w^2 collapses correlation to >0.99 within just a few layers (ordered")
print(f"phase); at sigma_w^2=3.0, correlation never approaches 1 even after 20 layers")
print(f"(chaotic phase) -- a genuine qualitative phase difference, not just a quantitative one.")

## 🤖 Why This Matters for AI

This is the theory behind principled weight initialization schemes used in virtually every deep
network today — **He initialization** ($\sigma_w^2=2/n$, for ReLU) and **Glorot/Xavier
initialization** ($\sigma_w^2 = 1/n$ or $2/(n_{in}+n_{out})$, originally derived for
$\tanh$/sigmoid) are both *exactly* the specific $\sigma_w^2$ values that keep signal
propagation stable — avoiding the vanishing/exploding variance this notebook demonstrated
directly above, and sitting at or near the edge-of-chaos correlation-map transition. This
isn't just a story about activations at initialization: the same forward-propagation
instabilities have a mirror-image backward-propagation counterpart in the gradients (recall the
vanishing-gradient problem from `06`), so getting initialization right is directly what makes a
given depth *trainable at all*, before a single gradient step is taken. Below, we close the loop
started in the Theory section: train real, deep $\tanh$ networks at several $\sigma_w^2$ values
on an actual classification task, and confirm directly that a network initialized deep in the
ordered phase fails to learn (gradient near zero, loss stuck), while initializations at or past
the edge of chaos train successfully to near-zero loss.

In [ ]:
torch.manual_seed(0)

def make_tanh_mlp(width, depth, sigma_w_sq, sigma_b_sq, seed):
    torch.manual_seed(seed)
    layers = []
    d_in = 10
    for _ in range(depth):
        lin = nn.Linear(d_in, width)
        with torch.no_grad():
            lin.weight.normal_(0, np.sqrt(sigma_w_sq / d_in))
            lin.bias.normal_(0, np.sqrt(sigma_b_sq))
        layers += [lin, nn.Tanh()]
        d_in = width
    out = nn.Linear(d_in, 1)
    with torch.no_grad():
        out.weight.normal_(0, np.sqrt(sigma_w_sq / d_in))
        out.bias.zero_()
    layers.append(out)
    return nn.Sequential(*layers)

rng_ai = np.random.default_rng(0)
X_ai = torch.tensor(rng_ai.normal(size=(200, 10)), dtype=torch.float32)
w_true_ai = rng_ai.normal(size=10)
y_ai = torch.tensor((X_ai.numpy() @ w_true_ai > 0).astype(np.float32)).reshape(-1, 1)

def train_and_track(sigma_w_sq, depth=15, width=64, seed=1, n_steps=150):
    model = make_tanh_mlp(width, depth, sigma_w_sq, 0.05, seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    losses, first_layer_grad_norms = [], []
    for step in range(n_steps):
        opt.zero_grad()
        out = model(X_ai)
        loss = loss_fn(out, y_ai)
        loss.backward()
        first_layer_grad_norms.append(model[0].weight.grad.norm().item())
        opt.step()
        losses.append(loss.item())
    return losses, first_layer_grad_norms

sigma_values = [0.5, 1.0, 1.3, 3.0]  # spans deep-ordered, near-critical, and chaotic phases
results = {}
for sigma_w_sq in sigma_values:
    losses, grad_norms = train_and_track(sigma_w_sq)
    results[sigma_w_sq] = (losses, grad_norms)
    print(f"sigma_w^2={sigma_w_sq}: initial loss={losses[0]:.4f}, final loss={losses[-1]:.4f}, "
          f"first-layer grad norm at step 0: {grad_norms[0]:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {0.5: "#C44E52", 1.0: "#DD8452", 1.3: "#55A868", 3.0: "#4C72B0"}
for sigma_w_sq in sigma_values:
    losses, grad_norms = results[sigma_w_sq]
    axes[0].plot(losses, color=colors[sigma_w_sq], lw=2, label=f"$\\sigma_w^2$={sigma_w_sq}")
    axes[1].semilogy(grad_norms, color=colors[sigma_w_sq], lw=2, label=f"$\\sigma_w^2$={sigma_w_sq}")
axes[0].set_xlabel("training step"); axes[0].set_ylabel("loss")
axes[0].set_title("Training loss")
axes[1].set_xlabel("training step"); axes[1].set_ylabel("first-layer gradient norm (log scale)")
axes[1].set_title("Gradient magnitude reaching the FIRST layer\nof a 15-layer network")
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nsigma_w^2=0.5 (deep in the ordered phase, per the correlation-map results above) has a")
print(f"first-layer gradient that stays tiny throughout training -- the network never learns")
print(f"(loss stuck near {results[0.5][0][-1]:.3f}, close to random-guessing's ln(2)~0.693).")
print(f"Every setting at or past the edge of chaos trains successfully to near-zero loss.")

## 🏋️ Exercises

### Warm-up
1. In the Implementation section, `var_low` (sigma_w^2=0.5) reaches a variance of exactly
   `0.0` by some layer. Explain in words why floating-point underflow (not a bug) is the actual
   cause: what happens to `np.var(x)` once `x`'s entries are all extremely close to zero, and
   why does ReLU make this a one-way trap (once activations collapse near zero, can they
   recover in later layers)?

### Practice
2. In the correlation-map experiment, change `c0` (the starting correlation between the two
   inputs) from `0.3` to `0.05` (nearly uncorrelated to begin with) and re-run for
   `sigma_w^2=0.8` and `sigma_w^2=3.0`. Does the ordered-phase setting still drive correlation
   to near 1, and does the chaotic-phase setting still keep it well below 1? What does this tell
   you about whether the LONG-RUN phase (ordered vs. chaotic) depends on where you start, versus
   only on $\sigma_w^2$ itself?

### Challenge
3. In the AI section, add a 5th setting, `sigma_w^2=1.05` (very close to the edge-of-chaos value
   suggested by the correlation-map results in the Implementation section), and increase `depth`
   from `15` to `30`. Does this near-critical setting train successfully, train slowly, or fail,
   compared to `sigma_w^2=1.3`? Relate your finding to why real deep networks typically pick an
   initialization comfortably on the chaotic side of the transition (or use complementary tricks
   like residual connections and normalization layers, previewed in `06`) rather than tuning
   $\sigma_w^2$ to sit exactly at the critical point.

---

## 📚 Further Reading
- Poole et al., [\"Exponential expressivity in deep neural networks through transient chaos\"](https://arxiv.org/abs/1606.05340)
- Schoenholz et al., [\"Deep Information Propagation\"](https://arxiv.org/abs/1611.01232)
- He et al., [\"Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification\"](https://arxiv.org/abs/1502.01852)

---

*Next notebook: [Capstone: Build Attention from Math](../12_Capstone_Projects/01_build_attention_from_math.ipynb)*